<h1>Heat of Formation Calculator Using Benson Group Increments</h1>
<h4>Designed for Chemistry C450/C540 at Indiana University Bloomington by Prashant Kumar and Nicola L. B. Pohl</h4><br>
September 12, 2024 Version<br>
<ul>
<li>This notebook is designed to calculate approximate heats of formation of organic molecules based on the idea of Benson Group Increments: Cohen & Benson, <em>Chem. Rev.</em> <b>1993</b>, <em>93</em>, 2419.</li>
<li>The notebook takes values from a user's file titled "increment_correction_table.csv" that is in the same folder as this notebook file. This csv file contains one column titled "Benson Group Increment" populated by Benson Group increment types and a second column titled "Delta_Hf kJ/mol" populated by numerical values in kJ/mol. Research is ongoing to update values and add increments to better describe a range of molecules; the user can decide which increments to use in the calculator.</li>
<li>The Benson Group Increments are rendered into compact buttons that the user can click on to select. The value associated with that button is automatically added to the total displayed below the buttons. The user needs to decide which increments and corrections are needed based on the molecule of interest to make the entire process of estimation transparent.</li> 
</ul>

In [ ]:
# --- Constants ---
# Conversion factor from kilojoules to kilocalories (source: NIST)
KJ_TO_KCAL = 0.239006

# --- Imports ---
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# --- Data Loading ---
def load_increment_data(csv_path):
    """Load and clean increment/correction data from CSV."""
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df.fillna(0, inplace=True)
    return df

def get_value_dicts(df):
    """Extract value dictionaries for CH, CHO, and corrections from DataFrame."""
    ch_dict = dict(zip(df['CH Benson Group Increment'], df['Delta_Hf kJ/mol']))
    cho_dict = dict(zip(df['CHO Benson Group Increment'], df['CHO Values']))
    corr_dict = dict(zip(df['Correction'], df['Correction Values']))
    return ch_dict, cho_dict, corr_dict

# --- Widget Creation ---
def create_increment_button(label, value, color, on_click_fn):
    """Create a compact button for a group increment or correction."""
    if label == '' or value == '':
        return None
    button = widgets.Button(description=f"{label}", layout=widgets.Layout(width="125px", height="25px", padding="0px 0px 0px 0px"))
    button.style.button_color = color
    button.on_click(lambda b: on_click_fn(value, label))
    return button

def create_button_grid(buttons, columns=6):
    """Create a grid layout for a list of buttons."""
    return widgets.GridBox([btn for btn in buttons if btn], layout=widgets.Layout(grid_template_columns=f"repeat({columns}, 150px)", grid_gap="2px"))

# --- State & UI ---
total_heat_kj = 0.0
increment_history = []
selected_button_labels = []

# Add a history panel output widget
history_panel = widgets.VBox()

total_label_kj = widgets.Label()
total_label_kcal = widgets.Label()
selected_buttons_label = widgets.Label()

def update_labels():
    total_label_kj.value = f"Standard heat of formation: {total_heat_kj:.2f} kJ/mol"
    total_label_kcal.value = f"Standard heat of formation: {total_heat_kj * KJ_TO_KCAL:.2f} kcal/mol"
    selected_buttons_label.value = f"Pressed buttons: {', '.join(selected_button_labels) if selected_button_labels else 'None'}"

def update_history_panel():
    """Redraw the history panel with current increment history and remove buttons."""
    import functools
    if not increment_history:
        history_panel.children = [widgets.HTML('<b>No increments added yet.</b>')]
    else:
        items = []
        for idx, (label, value) in enumerate(zip(selected_button_labels, increment_history)):
            remove_btn = widgets.Button(description='Remove', layout=widgets.Layout(width='70px', height='22px'))
            remove_btn.on_click(functools.partial(remove_history_item, idx))
            item_box = widgets.HBox([widgets.Label(f'{label}: {value:+.2f} kJ/mol'), remove_btn])
            items.append(item_box)
        history_panel.children = items

def remove_history_item(index, button=None):
    """Remove an item from history by index and update state/UI."""
    global total_heat_kj
    if 0 <= index < len(increment_history):
        total_heat_kj -= increment_history[index]
        del increment_history[index]
        del selected_button_labels[index]
        update_labels()
        update_history_panel()

# --- Event Handlers ---
def add_to_total(value, label):
    global total_heat_kj
    total_heat_kj += value
    increment_history.append(value)
    selected_button_labels.append(label)
    update_labels()
    update_history_panel()

def undo_last_action(button):
    global total_heat_kj
    if increment_history and selected_button_labels:
        last_value = increment_history.pop()
        total_heat_kj -= last_value
        selected_button_labels.pop()
        update_labels()
        update_history_panel()

def reset_all(button):
    global total_heat_kj, increment_history, selected_button_labels
    total_heat_kj = 0.0
    increment_history = []
    selected_button_labels = []
    update_labels()
    update_history_panel()

# --- Main UI Assembly ---
def main():
    df = load_increment_data('CSV_data_files/increment_correction_table.csv')
    ch_dict, cho_dict, corr_dict = get_value_dicts(df)

    ch_buttons = [create_increment_button(label, value, 'lightblue', add_to_total) for label, value in ch_dict.items() if label and value]
    cho_buttons = [create_increment_button(label, value, 'lightpink', add_to_total) for label, value in cho_dict.items() if label and value]
    corr_buttons = [create_increment_button(label, value, 'lightgreen', add_to_total) for label, value in corr_dict.items() if label and value]

    ch_layout = create_button_grid(ch_buttons)
    cho_layout = create_button_grid(cho_buttons)
    corr_layout = create_button_grid(corr_buttons)

    undo_button = widgets.Button(description="Undo the last addition", layout=widgets.Layout(width="440px", height="25px"))
    undo_button.on_click(undo_last_action)
    reset_button = widgets.Button(description="Reset all selections", layout=widgets.Layout(width="440px", height="25px"))
    reset_button.on_click(reset_all)
    button_container = widgets.HBox([undo_button, reset_button])

    tab = widgets.Tab()
    tab.children = [ch_layout, cho_layout, corr_layout]
    tab.set_title(0, 'CH Groups')
    tab.set_title(1, 'CHO Groups')
    tab.set_title(2, 'Corrections')

    update_labels()
    update_history_panel()
    display(tab)
    display(button_container)
    display(total_label_kj)
    display(total_label_kcal)
    display(selected_buttons_label)
    display(history_panel)

main()

FileNotFoundError: [Errno 2] No such file or directory: 'increment_correction_table.csv'